In [ ]:
# Ejecutar una vez por target, con permisos de administrador de Unity Catalog.
# Los tres grupos DEBEN ser grupos de cuenta (no CREATE GROUP de SQL).
dbutils.widgets.text('catalog','electrocasa_dev')
dbutils.widgets.text('account_id','')
dbutils.widgets.text('account_host','https://accounts.azuredatabricks.net')
catalog=dbutils.widgets.get('catalog')
assert catalog in ('electrocasa_dev','electrocasa')
for sql in [f'CREATE CATALOG IF NOT EXISTS `{catalog}`',
            *[f'CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{s}`' for s in ('bronze','silver','gold')],
            f'CREATE VOLUME IF NOT EXISTS `{catalog}`.`bronze`.`landing`',
            f'CREATE VOLUME IF NOT EXISTS `{catalog}`.`bronze`.`pipeline_state`']:
    spark.sql(sql)
print('Catálogo, esquemas y Volume preparados:',catalog)


In [ ]:
# Provisiona los grupos de cuenta. Requiere autenticación de administrador de CUENTA
# ya configurada para AccountClient; no almacenes un token en este notebook.
from databricks.sdk import AccountClient
from databricks.sdk.service.iam import Group
account_id=dbutils.widgets.get('account_id')
if not account_id:
    raise ValueError('Indica account_id y configura autenticación de administrador de cuenta antes de conceder permisos UC')
a=AccountClient(host=dbutils.widgets.get('account_host'),account_id=account_id)
for suffix in ('ingenieria','analistas','auditoria'):
    name=f'{catalog}_{suffix}'
    existing=list(a.groups.list(filter=f'displayName eq "{name}"'))
    if not existing:
        a.groups.create(display_name=name)
    assert list(a.groups.list(filter=f'displayName eq "{name}"')),name
print('Grupos de cuenta creados/verificados')


In [ ]:
# El ejecutor necesita ser dueño del catálogo o contar con MANAGE/GRANT.
eng=f'{catalog}_ingenieria'; ana=f'{catalog}_analistas'; aud=f'{catalog}_auditoria'
statements=[f'GRANT USE CATALOG ON CATALOG `{catalog}` TO `{g}`' for g in (eng,ana,aud)]
statements += [f'GRANT USE SCHEMA, SELECT, CREATE TABLE, CREATE MATERIALIZED VIEW, CREATE FUNCTION ON SCHEMA `{catalog}`.`{s}` TO `{eng}`' for s in ('bronze','silver','gold')]
statements += [f'GRANT USE SCHEMA, SELECT ON SCHEMA `{catalog}`.`gold` TO `{g}`' for g in (ana,aud)]
statements += [f'GRANT READ VOLUME, WRITE VOLUME ON VOLUME `{catalog}`.`bronze`.`{v}` TO `{eng}`' for v in ('landing','pipeline_state')]
for statement in statements: spark.sql(statement)
print('Permisos diferenciados aplicados; confirmar memberships reales antes de publicar')
